## Advanced Observable Expectation Values

This notebook demonstrates an advanced average value estimator for quantum observables. It is designed to be fully self-contained, including the key statistical equations used for simultaneous measurement and uncertainty quantification.

### Core Concepts

1. **Simultaneous Measurements**: We group Pauli operators into commuting sets (cliques) to measure them with the same circuit.
2. **Expectation Value**: For an observable $A = \sum a_i P_i$, the estimated mean is:
   $$\langle A \rangle = \sum a_i \langle P_i \rangle$$
3. **Uncertainty Tracking**: We track the full covariance matrix to compute the total variance:
   $$\text{Var}(A) = \sum_{i,j} a_i a_j \text{Covar}(P_i, P_j)$$
4. **Adaptive Optimization**: We iteratively redistribute measurement shots to minimize $\text{Var}(A)$.

In [83]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import SamplerV2 as Sampler

from advanced_estimation.commutation.general_commuting import GeneralCommutation
from advanced_estimation.estimation.pauli_estimation import (
    estimate_cliques_expectation_values_and_covariances,
    overall_paulis_expectation_values_and_covariances,
)

In [84]:
# Define the problem Hamiltonian (Target Observable)
observable = SparsePauliOp(
    ["XX", "YY", "ZZ", "IZ"],
    coeffs=[0.5, -0.5, 1.0, 0.2]
)

# State preparation (e.g., an entangled Bell state)
state_circuit = QuantumCircuit(2)
state_circuit.h(0)
state_circuit.cx(0, 1)

commutation_module = GeneralCommutation()
shots_budget = 2000

simulator = AerSimulator()
sampler = Sampler(mode=simulator)

print(f"Preparation Circuit:\n{state_circuit}")

Preparation Circuit:
     ┌───┐     
q_0: ┤ H ├──■──
     └───┘┌─┴─┐
q_1: ─────┤ X ├
          └───┘


## Step 1: Identifying Commuting Groups (Cliques)

Two Pauli strings $P_i$ and $P_j$ commute if $[P_i, P_j] = 0$. Using graph theory, we find cliques $A_u$ where all operators commute. 

This allows us to diagonalize them together using a Clifford transformation $R_u$:
$$R_u P_i R_u^\dagger = Z_i$$

In [85]:
paulis = observable.paulis
cliques_paulis_indices = commutation_module.find_commuting_cliques(paulis)

print(f"\nTotal Measurement Cliques: {len(cliques_paulis_indices)}")
for i, clique in enumerate(cliques_paulis_indices):
    strings = [str(paulis[idx]) for idx in clique]
    print(f"  Clique {i+1} : {strings}")


Total Measurement Cliques: 2
  Clique 1 : ['ZZ', 'XX', 'YY']
  Clique 2 : ['ZZ', 'IZ']


## Step 2: Resource Allocation

We start with a uniform distribution of measurement shots $M_u$ among the $U$ cliques:
$$M_u = \frac{\text{Budget}}{U}$$

This initial run provides the baseline data needed for optimization.

In [86]:
num_cliques = len(cliques_paulis_indices)
cliques_shots = [max(shots_budget // num_cliques, 1) for _ in range(num_cliques)]

print("Initial Shot Allocation:")
for i, s in enumerate(cliques_shots):
    print(f"  Clique {i+1}: {s} shots")

Initial Shot Allocation:
  Clique 1: 1000 shots
  Clique 2: 1000 shots


## Step 3a: Execution & Local Estimation

For each clique $u$, we sample bitstrings $b^{(m)}$ and estimate the local expectation value $E^{(u)}_\psi[P_i]$ and internal covariance $\text{Covar}^{(u)}_\psi[P_i, P_j]$:
$$E^{(u)}_\psi[P_i] = \frac{1}{M_u} \sum_{m=1}^{M_u} \langle b^{(m)} | Z_i | b^{(m)} \rangle$$
$$\text{Covar}^{(u)}_\psi[P_i, P_j] = E^{(u)}[Z_i Z_j] - E^{(u)}[Z_i] E^{(u)}[Z_j]$$

In [91]:
cliques_expectation_values, cliques_covariances = (
    estimate_cliques_expectation_values_and_covariances(
        paulis, cliques_paulis_indices, cliques_shots, 
        commutation_module, state_circuit, sampler
    )
)

print("\nEstimated Expectation Values for Each Clique:")
for i, (clique, exp_val) in enumerate(zip(cliques_paulis_indices, cliques_expectation_values)):
    strings = [str(paulis[idx]) for idx in clique]
    print(f"  Clique {i+1} ({strings}): {exp_val}")

print("\nCovariance Matrices for Each Clique:")
for i, (clique, cov) in enumerate(zip(cliques_paulis_indices, cliques_covariances)):
    strings = [str(paulis[idx]) for idx in clique]
    print(f"  Clique {i+1} ({strings}):\n{cov}\n")


Estimated Expectation Values for Each Clique:
  Clique 1 (['ZZ', 'XX', 'YY']): [ 1.  1. -1.]
  Clique 2 (['ZZ', 'IZ']): [1.    0.054]

Covariance Matrices for Each Clique:
  Clique 1 (['ZZ', 'XX', 'YY']):
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]

  Clique 2 (['ZZ', 'IZ']):
[[0.       0.      ]
 [0.       0.997084]]



## Step 3b: Integrated Global Synthesis

Since a Pauli operator might exist in multiple cliques, we combine the information. The global expectation for operator $i$ is (weighted by total shots $M_i$ for that operator):
$$\tilde{E}_\psi[P_i] = \frac{1}{M_i} \sum_{u} M_{u} E^{(u)}_\psi[P_i]$$

The integrated covariance between $P_i$ and $P_j$ is:
$$\text{Covar}_\psi[P_i, P_j] = \frac{M}{M_i M_j} \sum_{u \in \{i,j\}} M_u \text{Covar}^{(u)}_\psi[P_i, P_j]$$

In [88]:
num_paulis = paulis.size
paulis_expectation_values, paulis_covariances = (
    overall_paulis_expectation_values_and_covariances(
        num_paulis, cliques_paulis_indices, 
        cliques_expectation_values, cliques_covariances, cliques_shots
    )
)

print("Integrated Global Pauli Expectations:")
for p, val in zip(paulis, paulis_expectation_values):
    print(f"  <{p}> = {val:.4f}")

Integrated Global Pauli Expectations:
  <XX> = 1.0000
  <YY> = -1.0000
  <ZZ> = 1.0000
  <IZ> = 0.0340


## Step 4: Final Energy & Precision Measurement

We calculate the average energy and the **Total Precision** (Variance):
$$\text{Var}(A) = \sum_{i,j} a_i a_j \text{Covar}(P_i, P_j)$$

Note: $\text{Var}(A)$ decreases with the number of samples $M$ following $\text{Var} \propto \frac{1}{M}$.

In [89]:
coeffs = observable.coeffs.real
actual_tot_shots = sum(cliques_shots)

obs_exp = np.matmul(coeffs, paulis_expectation_values)
obs_var = np.einsum("ij,i,j", paulis_covariances, coeffs, coeffs) / actual_tot_shots

print(f"Energy Estimate: {obs_exp:.4f}")
print(f"Total Variance: {obs_var:.6f}")
print(f"Standard Error: {np.sqrt(obs_var):.6f}")

Energy Estimate: 2.0068
Total Variance: 0.000040
Standard Error: 0.006321


## Step 5: Adaptive Redistribution Heuristic

To minimize the variance for the next iteration, we calculate how much each clique $u$ contributes to the weighted variance:
$$\omega_u = \sum_{i,j \in A_u} a_i a_j \text{Covar}^{(u)}_\psi[P_i, P_j]$$

The optimal redistribution for the shots budget follows a **Square Root scaling**:
$$M_u^{\text{new}} \propto \sqrt{\omega_u}$$

This strategy moves the budget toward cliques that are statistically 'harder' to estimate (more noise), reducing global error efficiently.

In [90]:
weighted_cliques_variances = []
for indices, cov_matrix in zip(cliques_paulis_indices, cliques_covariances):
    clique_coeffs = coeffs[indices]
    w_var = np.einsum("jk,j,k", cov_matrix, clique_coeffs, clique_coeffs)
    weighted_cliques_variances.append(w_var)

weighted_cliques_stds = np.sqrt(np.array(weighted_cliques_variances))

new_cliques_shots = np.ceil(
    weighted_cliques_stds / np.sum(weighted_cliques_stds) * shots_budget
).astype(int)
new_cliques_shots[new_cliques_shots == 0] = 1

print("Strategic Budget Shift (Next Iteration):")
for i, (old, new, w_v) in enumerate(zip(cliques_shots, new_cliques_shots, weighted_cliques_variances)):
    print(f"  Clique {i+1} : {old} -> {new} shots (Weighted Var: {w_v:.4f})")

Strategic Budget Shift (Next Iteration):
  Clique 1 : 1000 -> 1 shots (Weighted Var: 0.0000)
  Clique 2 : 1000 -> 2000 shots (Weighted Var: 0.0400)
